# Chapter 6 Object, Reference, Mutability and Recycling

## Topic 1: Variables Are Not Boxes (Variables as Labels)
1. What It Is
- In introductory programming, variables are often explained using the metaphor of a box that stores data. In Python, this metaphor is actively misleading.
- A Python variable is not a box; it is ***a label*** (or a sticky note with a name on it) attached to an object. Objects exist **independently** in memory, and variable names are bound to those objects.

2. The Mechanism: Evaluation and Binding Order
- To understand how Python processes assignments, you must look at the two sides of the `=` operator:
    - Right-Hand Side (RHS) First: Python always evaluates the right-hand side expression first. This creates a new object in memory or retrieves an existing one.
    - Left-Hand Side (LHS) Binding Second: Only after the object on the RHS has been successfully instantiated/evaluated does Python bind (attach) the variable label on the LHS to that object.

3. Gotchas & Edge Cases
- Aliasing Illusion: Writing `a = [1, 2, 3]` followed by `b = a` does not copy the list. It attaches a second label `b` to the exact same list object referenced by a. Mutating `a.append(4)` makes the change visible through `b` because both are **aliases** for ***one underlying object***.
- Improper Terminology: Saying "the object is assigned to x" is backwards. The precise mental model is: "the variable x is assigned (bound) to the object"
- `a = a + [4]` will first evaluate the RHS by creating a new list object and then binds the variable `a` to this new list; so after this line `a` is now attached to a different object than before!
- `a += [4]` this is different because `+=` is an in-place mutation meaning it mutates the variable `a` in place rather than creating a new object so after this line `a` still binds to the same object as before and this object is mutated!
    - For mutable objects such as lists, `a + b` creates a new result object and rebinds `a`, while `a += b` usually mutates the existing object in place, preserving aliases to that object. For **immutable** objects such as tuples, `+=` cannot mutate the original object, so it creates a **new** object and rebinds the left-hand-side variable.

In [1]:
class Counter:
    def __init__(self, start):
        self.val = start
        print(f"Created Counter({start})")
    
    def add(self, n):
        if n < 0:
            raise ValueError("Negative addition not allowed")
        self.val += n
        return self

c1 = Counter(10)
c2 = c1

try:
    c3 = Counter(5).add(-2)
except ValueError:
    pass

c1.add(5)

# Print #1
print(1, c2.val) # 15

# Print #2
print(2, 'c3' in locals()) # False

Created Counter(10)
Created Counter(5)
1 15
2 False


In [ ]:
items = [1, 2]
ref = items
items = items + [3] #NOTE: This rebinds the variable into a new object created on the RHS.
print(ref is items)
items, ref

False


([1, 2, 3], [1, 2])

In [ ]:
items2 = [1, 2]
ref2 = items2
items2 += [3] #NOTE: this is an in-place mutation, it does NOT rebind the variable!
items2, ref2, ref2 is items2

([1, 2, 3], [1, 2, 3], True)

In [ ]:
items3 = (1, 2)
ref3 = items3
items3 += (3,) #NOTE: This creates a new tuple as tuple is immutable so can't do in place mutation; then bind to items3
ref3 is items3 # False
print(ref3, items3)

(1, 2) (1, 2, 3)


In [18]:
t = (1, 2, [3, 4])
#NOTE: here it does in place mutation on a list that is part of a tuple.
# The mutation works but binding the updated list to the 3rd item of a tuple fails due to tuple immutability. 
try:
    t[2] += [5] 
except TypeError as e:
    print(e)
    print(t[2])

'tuple' object does not support item assignment
[3, 4, 5]


## Topic 2: Identity, Equality, and Aliasing (is vs ==)
1. What It Is
- Every Python object has three core traits: an identity, a type, and a value. Type and identity are fixed from the moment of creation and can never change. Only the value (the data held inside the object) may change over time.
- Aliasing occurs when two or more variables are bound to the exact same object in memory.

2. The Mechanism: Under the Hood of `is` and `==`
- A. Identity (`is` operator)
The `is` operator compares the identities of two objects. Under the hood: In CPython, an object's identity is its raw memory address. You can inspect this integer address directly using `id(obj)`.
When Python evaluates `a is b`, it directly compares the two integer memory addresses at the C level. Because `is` compares memory addresses directly, it **cannot** be overloaded by custom methods. It executes at *high speed* as a simple integer comparison.
- B. Value Equality (`==` operator)
The `==` operator compares the values (contents) of two objects. Under the hood: The expression `a == b` is syntactic sugar for calling the special method `a.__eq__(b)`.
The default `__eq__` method inherited from the root object base class simply compares object IDs (falling back to is behavior).
However, built-in classes (like `dict`, `list`, `str`) override `__eq__` to recursively traverse and compare their underlying data attributes.

3. Gotchas & Best Practices
- Never Use `is` for Value Checks: Do not use `is` to check numbers, strings, or data values (e.g., `if name is "Alice"`). Interpreter optimizations (like string interning) can make equal strings share memory addresses unpredictably, leading to subtle bugs.
- The Singleton Sentinel Exception: The primary legitimate use of is in everyday Python is checking against singletons—most notably `None`.
    ```
    if x is None
    if x is not None
    ```
    Because `None` exists as a unique, single instance in memory throughout the Python interpreter session, checking `x is None` is faster and safer than `x == None`.

In [16]:
class User:
    def __init__(self, user_id, name):
        self.user_id = user_id
        self.name = name

    def __eq__(self, other):
        if isinstance(other, User):
            return self.user_id == other.user_id
        return False

u1 = User(101, "Alice")
u2 = User(101, "Alice")
u3 = u1
sentinel = None

# Print #1
print(1, u1 == u2, u1 is u2) # True False

# Print #2
print(2, u1 == u3, u1 is u3) # True True

# Print #3
print(3, u1 is sentinel, u1 == sentinel) # False False

1 True False
2 True True
3 False False


## Topic 4: Shallow Copies by Default
1. What It Is
- When you duplicate a collection using built-in constructors (such as `list(l1)`) or sequence slicing (`l1[:]`), Python creates a shallow copy.
- A shallow copy duplicates the outermost container, but populates it with references (pointers) to the exact same elements contained in the original collection.

2. The Mechanism: Shared References in Nested Structures
- To understand how a shallow copy behaves in memory, consider a nested sequence `l1 = [3, [1, 2], (7, 8)]` copied via `l2 = list(l1)` 
- Top-Level Independence: The outer list `l2` is a newly allocated list object in memory (`l2 is l1` evaluates to `False`). Appending or removing items directly at the top level of `l1` does not affect `l2`.
- Nested Aliasing: The elements inside `l1` and `l2` are copied by reference. Index 1 in both lists points to the exact same inner list object in memory (`l1[1] is l2[1]` evaluates to `True`).
- In-Place Mutation vs. Rebinding:
Mutating a shared nested object (e.g., `l2[1].append(99)`) alters the object in place, making the change immediately visible through both `l1` and `l2`.
Performing augmented assignment on an immutable nested object (e.g., `l2 += (10, 11)` on a tuple) creates a new tuple object and rebinds `l2` to that new address. `l1` remains bound to the original tuple.

3. Gotchas & Edge Cases
- The Slicing Shortcut Trap: Writing `l2 = l1[:]` behaves identically to `l2 = list(l1)` —it is still a shallow copy. It does not recursively duplicate nested mutable structures.
- Immutable Containers Are Shared: If all elements inside a collection are immutable (such as integers or strings), a shallow copy is completely safe because none of the elements can be mutated in place

In [ ]:
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)
l1.append(100)
l1[1].remove(55)
print('l1:', l1) # [3, [66, 44], (7, 8, 9), 100]
print('l2:', l2) # [3, [66, 44], (7, 8, 9)]
l2[1] += [33, 22] # l2 = [3, [66, 44, 33, 22], (7, 8, 9)]
l2[2] += (10, 11) # l2 = [3, [66, 44, 33, 22], (7, 8, 9, 10, 11)]
print('l1:', l1) # [3, [66, 44, 33, 22], (7, 8, 9), 100]
print('l2:', l2) # [3, [66, 44, 33, 22], (7, 8, 9, 10, 11)]

l1: [3, [66, 44], (7, 8, 9), 100]
l2: [3, [66, 44], (7, 8, 9)]
l1: [3, [66, 44, 33, 22], (7, 8, 9), 100]
l2: [3, [66, 44, 33, 22], (7, 8, 9, 10, 11)]


In [20]:
a = [1, [5, 6], (30, 40)]
b = list(a)  # Shallow copy

# Operation 1: Append to top-level list 'a'
a.append(99) # a = [1, [5, 6], (30, 40), 99]

# Operation 2: Mutate nested list via 'b'
b[1].append(30) # b = [1, [5, 6, 30], (30, 40)], a = [1, [5, 6, 30], (30, 40), 99]

# Operation 3: Augmented assignment on nested tuple via 'b'
b[2] += (50, 60) # b = [1, [5, 6, 30], (30, 40, 50, 60)]

# Print #1
print(1, a) # [1, [5, 6, 30], (30, 40), 99]

# Print #2
print(2, b) # [ 1, [5, 6, 30], (30, 40, 50, 60)]

# Print #3
print(3, a[2] is b[2]) # False

1 [1, [5, 6, 30], (30, 40), 99]
2 [1, [5, 6, 30], (30, 40, 50, 60)]
3 False


## Topic 5: Deep and Shallow Copies of Arbitrary Objects (copy vs deepcopy)

1. What It Is
- The copy module provides generic copy operations for arbitrary user-defined objects and nested structures via two functions:
    - copy.copy(obj): Creates a shallow copy of obj.
    - copy.deepcopy(obj): Creates a deep copy of obj, recursively duplicating all nested objects so no mutable references are shared.

2. The Mechanism: Reference Tracking & Cycle Handling
Under the hood, duplicating custom objects involves two key mechanisms:
- A. Attribute Dictionary Copying
When copy.copy(bus1) is called on a custom class instance, Python creates a new instance of the class and populates its internal `__dict__` with shallow copies of the original instance's attribute references. 
- B. Cycle Detection in deepcopy
A naive recursive algorithm for deep copies would enter an infinite recursion loop if an object contained a cyclic reference (e.g., list a containing list b, where b also contains a reference back to a). To prevent this, copy.deepcopy maintains a memoization dictionary during traversal:
    - Before copying an object, deepcopy checks its internal memo table using the object's id().
    - If the object has already been copied during the current operation, deepcopy simply retrieves the existing reference from the memo table instead of recursing again.

3. Gotchas & Customizing Copy Behavior
Over-Deep Copying: A deep copy can sometimes be too deep. If an object references external system resources (e.g., file descriptors, database connections) or singletons, duplicating them can crash or corrupt external state.
Hooks for Custom Classes: You can override copy behavior on custom classes by implementing the `__copy__()` and `__deepcopy__(self, memo)` special methods. The `__deepcopy__` method must accept a memo dictionary to preserve cycle detection.

In [22]:
a = [10, 20]
b = [a, 30]
a.append(b)
a

[10, 20, [[...], 30]]

In [27]:
import copy

class Node:
    def __init__(self, value):
        self.value = value
        self.children = []

# Setup a tree-like structure
parent = Node("Parent")
child = Node("Child")
parent.children.append(child)

# Create copies
shallow_parent = copy.copy(parent)
deep_parent = copy.deepcopy(parent)

# Mutate the original child node directly
child.value = "Updated Child"
child.children.append(Node("Grandchild"))

# Print #1
print(1, shallow_parent.children[0].value) # Updated Child

# Print #2
print(2, deep_parent.children[0].value) # Child

# Print #3
print(3, len(shallow_parent.children[0].children), len(deep_parent.children[0].children)) # 1, 0

1 Updated Child
2 Child
3 1 0


## Topic 6: Function Parameters as References
1. Call by sharing is the only mode of parameter passing in Python and it means that each formal parameter of the function gets a **copy of each reference in the arguments**. In other words, the parameters inside the function become **aliases** of the actual arguments.
    - Python function parameters are **ALWAYS** references!

2. The result of this scheme is that a function may change any ***mutable object passed as a parameter***, but it cannot change the identity of those objects (i.e., it cannot altogether replace an object with another).

3. This also means mutable types as parameter default is a bad idea! 
    - The problem is that **each default value is evaluated when the function is defined** i.e., usually when the module is loaded and the default values become attributes of the function object. So if a default value is a mutable object, and you change it, the change will affect every ***future*** call of the function.

4. Defensive Programming with Mutable Parameters
- When you are coding a function that receives a mutable parameter, you should carefully consider whether the caller expects the argument passed to be **changed**. It's possible that the caller/client expects the argument to be changed but when in doubt make a copy of the parameter to be defensive. Though making a copy is not free (CPU and memory).

5. del and Garbage Collection
    - `del` is a statement NOT a function though `del(x)` works, that's because `x` and `(x)` are usually the same thing in Python!
    - `del` deletes references NOT objects! GC may discard the object from memory if its last reference was deleted or rebound.
    - There is `__del__()` that is intended to be called by the interpreter when the instance is about to be deleted to give it an chance to release external resources. You should seldom need to implement it!
    - In CPython, the primary algorithm for GC is reference counting, essentially as soon as the `refcount` reaches 0, the object gets destroyed. In CPython2, it adds a generational GC algorithm to handle objects in reference cycle.
    - Weak references don't prevent objects being GCed and they're useful in caching applications because you don't want the cached objects to be kept alive just because they're referenced by the cache. 
6. Surprisingly `tuple(t)` when `t` is a tuple actually creates a new referencing to the same object as `t` does, despite tuple being immutable.
    - The same is also true for other immutable types: `str`, `bytes` and `frozenset`.
    - Those should be harmless as they're immutables; CPython tries to save memory and makes the interpreter faster.

In [34]:
def f(a, b):
    a += b
    return a
a, b = 1, 2
print(f(a, b))
a, b #immutables are unchanged

3


(1, 2)

In [35]:
x, y = [1, 2], [3, 4]
print(f'x={x}, y={y}')
print(f'f(x, y)={f(x, y)}')
print(f'x={x}, y={y}') # x is mutable and is changed!

x=[1, 2], y=[3, 4]
f(x, y)=[1, 2, 3, 4]
x=[1, 2, 3, 4], y=[3, 4]


In [36]:
t, u = (1, 2), (3, 4)
print(f(t, u))
t, u # immutables are unchanged!

(1, 2, 3, 4)


((1, 2), (3, 4))

In [50]:
class BadBus:
    def __init__(self, passengers = []):
        self.passengers = passengers
    def pick(self, name):
        self.passengers.append(name)
    def drop(self, name):
        self.passengers.remove(name)

print(BadBus.__init__.__defaults__)

([],)


In [51]:
bad_bus1 = BadBus()
bad_bus1.pick('Tom') #NOTE: the default is changed and affects every future call!

bad_bus2 = BadBus()
print(BadBus.__init__.__defaults__) #NOTE: Here its default passengers is sharing the default with bad bus1.
print(BadBus.__init__.__defaults__[0] is bad_bus2.passengers)


(['Tom'],)
True


In [52]:
class GoodBus:
    def __init__(self, passengers = None):
        if passengers is None:
            self.passengers = []
        else:
            #self.passengers = passengers #NOTE: this would mutate the passed in passengers!
            self.passengers = list(passengers) #NOTE: make a copy!

In [ ]:
t1 = (1, 2)
t2 = tuple(t1)
print(t1 is t2) # NOTE: This is True!

s1 = 'hi'
s2 = 'hi'
print(s1 is s2) #NOTE: also True

fs1 = frozenset([1, 2, 3])
fs2 = frozenset(fs1)
fs3 = fs1.copy()
print(fs1 is fs2, fs1 is fs3) #NOTE both are True

True
True
True True
